[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/time_series.ipynb)

# Time series and forecasting full course: the practice set, in Python

Thirty-five exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the module's standard methods, several steps each), **tier 3** stretch problems (choose the
method, compare on a holdout, spot the trap, work backwards) and **tier 4**, the case: four years of monthly support
tickets at Tallowfield Software (a fictional company; our own data), and the question of how many support agents to
roster in each month of next year. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same methods with `pandas` and `numpy`
(a rolling mean, an exponentially weighted mean, a trend fit, least squares with dummy columns and a seasonal
decomposition), so you can run them on your own data. And two cells you can move yourself: a smoothing constant α
that redraws the forecast, and Holt's α and β on a series with a trend.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Percentages and
MAPE as plain numbers: `6.69` for 6.69% (typing the % sign is fine too). Numbers without thousands separators:
`25768.5`, not `25,768.5`. You can type the arithmetic too: `(64+70+67)/3`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). Letters as they are: `C`. You can
also put your answer between the quotes in a cell's last line, `practice("P1.6", "")`, and run the cell: it checks
straight away.

**The cheat sheet.** Every method, formula and spreadsheet function from the course on two pages:
[time series and forecasting cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/time_series_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, random, textwrap
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the data CHALK · forecasts SKY · trend lines VIOLET · errors CORAL · the number that matters GOLD · seasons MINT

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(units?|tickets?|agents?|agent-months?|months?|weeks?|days?|years?|hundreds?|thousands?|millions?)\b|[$%]"
SAME = {"c": {"c", "option c", "seasonal"}, "d": {"d", "option d", "cyclical"}}

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    if isinstance(want, str):
        g = got.lower() if isinstance(got, str) else f"{got:g}"
        return g == want.lower() or g in SAME.get(want.lower(), ())
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Time series patterns',
     "q": "A fuel distributor's heating-oil deliveries (thousand litres) by quarter, three years: 84, 41, 22, 73 · 86, 39, 24, 71 · 85, 40, 23, 72. Each year's total is the same. Which pattern is it? A horizontal, B trend, C seasonal, D cyclical.",
     "parts": ['type one letter: A, B, C or D'],
     "answer": ['C'], "tol": [0], "function": 'SUM',
     "working": "The same quarters are high (Q1, Q4) and low (Q2, Q3) every year, and the yearly totals don't grow (220 each): a seasonal pattern without trend."},
    {"id": 'P1.2', "tier": 1, "group": 'Time series patterns',
     "q": "A machine-tool maker's yearly orders ($ millions), years 1–8: 40, 44, 49, 52, 47, 45, 50, 56. Give the year-on-year changes. Which pattern is it? A horizontal, B trend only, C seasonal, D cyclical around a trend.",
     "parts": ['change, year 1 to 2', 'change, year 2 to 3', 'change, year 3 to 4', 'change, year 4 to 5', 'change, year 5 to 6', 'change, year 6 to 7', 'change, year 7 to 8', 'type one letter: A, B, C or D'],
     "answer": [4, 5, 3, -5, -2, 5, 6, 'D'], "tol": [0, 0, 0, 0, 0, 0, 0, 0], "function": '—',
     "working": 'Changes +4, +5, +3, −5, −2, +5, +6: three years up, two down, then up again, on a rising path. Runs longer than a year around the trend: cyclical'},
    {"id": 'P1.3', "tier": 1, "group": 'Forecast accuracy',
     "q": 'Actual demand was 120 units; the forecast was 112. Give the forecast error and the absolute percentage error.',
     "parts": ['forecast error (units)', 'absolute percentage error (%, as a number)'],
     "answer": [8, 6.666666666666667], "tol": [0, 0.005], "function": 'ABS',
     "working": 'error = actual − forecast = 120 − 112 = 8; |8|/120 = 6.67 %'},
    {"id": 'P1.4', "tier": 1, "group": 'Forecast accuracy',
     "q": 'Four forecast errors: 3, −5, 2, −4. Give the mean error and the MAE.',
     "parts": ['mean error', 'MAE'],
     "answer": [-1.0, 3.5], "tol": [0, 0], "function": 'AVERAGE, ABS',
     "working": 'mean error = −4/4 = −1; MAE = (3 + 5 + 2 + 4)/4 = 3.5'},
    {"id": 'P1.5', "tier": 1, "group": 'Forecast accuracy',
     "q": 'The same errors, 3, −5, 2, −4: give the MSE and the RMSE.',
     "parts": ['MSE', 'RMSE'],
     "answer": [13.5, 3.6742346141747673], "tol": [0, 0.0005], "function": 'AVERAGE, SQRT',
     "working": 'MSE = (9 + 25 + 4 + 16)/4 = 13.5; RMSE = √13.5 ≈ 3.674'},
    {"id": 'P1.6', "tier": 1, "group": 'Moving averages and exponential smoothing',
     "q": "The last three weeks' orders were 64, 70 and 67. What is the three-week moving-average forecast for next week?",
     "parts": ['forecast for next week'],
     "answer": [67.0], "tol": [0], "function": 'AVERAGE',
     "working": '(64 + 70 + 67)/3 = 201/3 = 67'},
    {"id": 'P1.7', "tier": 1, "group": 'Moving averages and exponential smoothing',
     "q": "Exponential smoothing with α = 0.25: this week's forecast was 200 and the actual was 212. What is next week's forecast?",
     "parts": ["next week's forecast"],
     "answer": [203.0], "tol": [0], "function": '—',
     "working": 'F = αY + (1 − α)F = 0.25 × 212 + 0.75 × 200 = 53 + 150 = 203 (or 200 + 0.25 × 12)'},
    {"id": 'P1.8', "tier": 1, "group": 'Trend projection',
     "q": 'A trend line fitted to ten years of data is T = 35 + 1.8t, with t = 1 for the first year. Forecast year 11.',
     "parts": ['forecast for year 11'],
     "answer": [54.8], "tol": [0.005], "function": 'FORECAST.LINEAR',
     "working": 'T(11) = 35 + 1.8 × 11 = 54.8'},
    {"id": 'P1.9', "tier": 1, "group": 'Seasonality and trend',
     "q": 'A no-trend quarterly model is Ŷ = 80 + 12 Qtr1 − 5 Qtr2 + 3 Qtr3 (the fourth quarter is the baseline). Forecast a second quarter, and a fourth quarter.',
     "parts": ['a second quarter', 'a fourth quarter'],
     "answer": [75, 80], "tol": [0, 0], "function": '—',
     "working": 'Second quarter: Qtr2 = 1, the others 0: 80 − 5 = 75. Fourth quarter: all dummies 0: 80'},
    {"id": 'P1.10', "tier": 1, "group": 'Time series decomposition',
     "q": "A quarter's sales were $46.2 million and its seasonal index is 1.10. What is the deseasonalised value?",
     "parts": ['deseasonalised value ($ millions)'],
     "answer": [42.0], "tol": [0.005], "function": '—',
     "working": '46.2/1.10 = 42.0: what the quarter would have been without its seasonal lift'},
    {"id": 'P2.1', "tier": 2, "group": 'Forecast accuracy',
     "q": 'Weekly orders: 40, 44, 42, 45, 48. Use the naive forecast (next week = this week) for weeks 2–5 and give its MAE, MSE and MAPE.',
     "parts": ['MAE', 'MSE', 'MAPE (%, as a number)'],
     "answer": [3.0, 9.5, 6.69237012987013], "tol": [0, 0, 0.005], "function": 'ABS, AVERAGE',
     "working": 'Errors 4, −2, 3, 3; MAE = 12/4 = 3; MSE = (16 + 4 + 9 + 9)/4 = 9.5; MAPE = (4/44 + 2/42 + 3/45 + 3/48)/4 ≈ 6.69 %'},
    {"id": 'P2.2', "tier": 2, "group": 'Moving averages and exponential smoothing',
     "q": "Daily output of a bottling line (thousand bottles): 36, 42, 39, 45, 42, 48. Give the three-day moving-average forecasts for days 4–7, and compare its MSE on days 4–6 with the naive forecast's.",
     "parts": ['moving average: forecast for day 4', 'for day 5', 'for day 6', 'for day 7', 'moving average: MSE on days 4–6', 'naive: MSE on days 4–6'],
     "answer": [39.0, 42.0, 42.0, 45.0, 24.0, 27.0], "tol": [0, 0, 0, 0, 0, 0], "function": 'AVERAGE',
     "working": 'MA: 39, 42, 42, then 45 for day 7. Errors days 4–6: 6, 0, 6 → MSE 24. Naive errors: 6, −3, 6 → MSE 27. The moving average is a little better here'},
    {"id": 'P2.3', "tier": 2, "group": 'Moving averages and exponential smoothing',
     "q": 'The same output, 36, 42, 39, 45, 42, 48. Forecast day 7 with weights 0.5, 0.3, 0.2 (newest first).',
     "parts": ['forecast for day 7'],
     "answer": [45.6], "tol": [0.005], "function": 'SUMPRODUCT',
     "working": '0.5 × 48 + 0.3 × 42 + 0.2 × 45 = 24 + 12.6 + 9 = 45.6'},
    {"id": 'P2.4', "tier": 2, "group": 'Moving averages and exponential smoothing',
     "q": 'Weekly loads shipped: 120, 126, 117, 129. Exponential smoothing with F2 = 120. With α = 0.3, give F3, F4, F5 and the MSE over weeks 2–4; then the MSE with α = 0.6. Which α is better here?',
     "parts": ['F3 (α = 0.3)', 'F4', 'F5', 'MSE with α = 0.3, weeks 2–4', 'MSE with α = 0.6, weeks 2–4'],
     "answer": [121.8, 120.35999999999999, 122.95199999999997, 44.56320000000008, 55.72319999999996], "tol": [0.005, 0.005, 0.005, 0.005, 0.005], "function": '—',
     "working": 'α = 0.3: F3 = 0.3 × 126 + 0.7 × 120 = 121.8; F4 = 120.36; F5 ≈ 122.95; errors 6, −4.8, 8.64 → MSE ≈ 44.56. α = 0.6: errors 6, −6.6, 9.36 → MSE ≈ 55.72. The smaller α wins on this noisy series'},
    {"id": 'P2.5', "tier": 2, "group": 'Trend projection',
     "q": "A firm's annual maintenance contracts (hundreds): 12, 15, 17, 18, 23, 26 for years 1–6. Fit the least-squares trend line and forecast year 7.",
     "parts": ['slope b₁', 'intercept b₀', 'forecast for year 7'],
     "answer": [2.7142857142857144, 9.0, 28.0], "tol": [0.0005, 0.0005, 0.005], "function": 'SLOPE, INTERCEPT, FORECAST.LINEAR',
     "working": 't̄ = 3.5, ȳ = 18.5; Σ(t − t̄)(y − ȳ) = 47.5, Σ(t − t̄)² = 17.5; b1 = 47.5/17.5 ≈ 2.714; b0 = 18.5 − 2.714 × 3.5 = 9.0; year 7: 9.0 + 2.714 × 7 = 28.0'},
    {"id": 'P2.6', "tier": 2, "group": 'Trend projection',
     "q": "Holt's method with α = 0.4 and β = 0.2. After last month: level 120, trend 4. This month's actual is 130. Update the level and the trend, then forecast one and three months ahead.",
     "parts": ['new level L', 'new trend b', 'forecast one month ahead', 'forecast three months ahead'],
     "answer": [126.39999999999999, 4.479999999999999, 130.88, 139.83999999999997], "tol": [0.005, 0.005, 0.005, 0.005], "function": '—',
     "working": 'Forecast was 124; L = 0.4 × 130 + 0.6 × 124 = 126.4; b = 0.2 × (126.4 − 120) + 0.8 × 4 = 4.48; one ahead 130.88; three ahead 126.4 + 3 × 4.48 = 139.84'},
    {"id": 'P2.7', "tier": 2, "group": 'Seasonality and trend',
     "q": "A ferry's passengers (thousands) by quarter, two years: 30, 48, 75, 35 · 34, 52, 79, 31. No trend. Fit Ŷ = b0 + b1 Qtr1 + b2 Qtr2 + b3 Qtr3 and forecast next year's third quarter.",
     "parts": ['b₀', 'b₁ (Qtr1)', 'b₂ (Qtr2)', 'b₃ (Qtr3)', "forecast: next year's third quarter"],
     "answer": [33.0, -1.0, 17.0, 44.0, 77.0], "tol": [0.005, 0.005, 0.005, 0.005, 0.005], "function": 'LINEST, AVERAGEIF',
     "working": "With no trend, each fitted value is that quarter's mean: 32, 50, 77, 33. So b0 = 33 (the fourth quarter), b1 = −1, b2 = 17, b3 = 44; third quarter: 33 + 44 = 77"},
    {"id": 'P2.8', "tier": 2, "group": 'Time series decomposition',
     "q": "The retailer's first five quarters ($ millions): 18.2, 21.9, 20.6, 26.8, 19.7. Give the two four-quarter moving averages, the centred moving average at quarter 3, and quarter 3's seasonal-irregular ratio.",
     "parts": ['first 4-quarter moving average', 'second 4-quarter moving average', 'centred moving average, quarter 3', 'seasonal-irregular ratio, quarter 3'],
     "answer": [21.875, 22.25, 22.0625, 0.9337110481586403], "tol": [0.005, 0.005, 0.005, 0.0005], "function": 'AVERAGE',
     "working": '(18.2 + 21.9 + 20.6 + 26.8)/4 = 21.875; (21.9 + 20.6 + 26.8 + 19.7)/4 = 22.25; centred (21.875 + 22.25)/2 = 22.0625; ratio 20.6/22.0625 ≈ 0.9337'},
    {"id": 'P2.9', "tier": 2, "group": 'Time series decomposition',
     "q": 'Seasonal-irregular ratios over three years: Q1 0.84, 0.86, 0.85 · Q2 1.04, 1.02, 1.03 · Q3 1.12, 1.10, 1.11 · Q4 1.06, 1.04, 1.05. Find the seasonal indexes, adjusted to average 1. Then deseasonalise a third quarter of 66.6.',
     "parts": ['seasonal index Q1', 'seasonal index Q2', 'seasonal index Q3', 'seasonal index Q4', 'the third quarter, deseasonalised'],
     "answer": [0.8415841584158416, 1.0198019801980198, 1.099009900990099, 1.0396039603960396, 60.59999999999999], "tol": [0.0005, 0.0005, 0.0005, 0.0005, 0.005], "function": 'AVERAGE',
     "working": 'Averages 0.85, 1.03, 1.11, 1.05 sum to 4.04; multiply each by 4/4.04: 0.8416, 1.0198, 1.0990, 1.0396; 66.6/1.0990 ≈ 60.6'},
    {"id": 'P3.1', "tier": 3, "group": 'Forecast accuracy',
     "q": "Two methods on a four-week holdout. Actual: 52, 47, 55, 50. Method A: 52, 47, 54, 41. Method B: 49, 44, 52, 47. Give each one's MAE and MSE. Which measure prefers which, and which would you use if one big miss means an empty shelf?",
     "parts": ['method A: MAE', 'method A: MSE', 'method B: MAE', 'method B: MSE'],
     "answer": [2.5, 20.5, 3.0, 9.0], "tol": [0, 0, 0, 0], "function": 'ABS, AVERAGE',
     "working": 'A: errors 0, 0, 1, 9 → MAE 2.5, MSE 20.5. B: 3, 3, 3, 3 → MAE 3, MSE 9. MAE prefers A, MSE prefers B. Squaring punishes the one big miss, so when a big miss is costly, judge by MSE and choose B'},
    {"id": 'P3.2', "tier": 3, "group": 'Forecast accuracy',
     "q": "Weekly part demand, weeks 1–8: 51, 46, 53, 48, 55, 47, 54, 49 (then 50, 45, 52, 56 in weeks 9–12). Method 1 forecasts the average of weeks 1–8. Method 2 is a least-squares curve with six coefficients (a degree-5 polynomial in t) through weeks 1–8. Give each one's MAE on weeks 1–8 (the fit) and on weeks 9–12 (the holdout).",
     "parts": ['average: MAE on weeks 1–8', 'curve: MAE on weeks 1–8', 'average: MAE on weeks 9–12', 'curve: MAE on weeks 9–12'],
     "answer": [2.875, 2.000000000000032, 3.25, 430.0795454545457], "tol": [0.005, 0.005, 0.005, 0.5], "function": 'AVERAGE, ABS',
     "working": "Fit: average ≈ 2.88, curve ≈ 2.0, so the curve looks better. Holdout: average 3.25, curve ≈ 430 (it forecasts about 12 parts in week 9 and minus 112.5 in week 10). Judge a method on data it hasn't seen"},
    {"id": 'P3.3', "tier": 3, "group": 'Moving averages and exponential smoothing',
     "q": "Exponential smoothing on the weekly part demand (F2 = 51). Choose α from 0.1, 0.3, 0.5, 0.7 by the MSE on weeks 2–8, then report the chosen α's MSE on the holdout, weeks 9–12.",
     "parts": ['the chosen α', 'its MSE on weeks 2–8', 'its MSE on weeks 9–12'],
     "answer": [0.1, 12.894362062589282, 17.21686311661713], "tol": [0, 0.005, 0.005], "function": '—',
     "working": 'Build MSEs: α 0.1: 12.89, α 0.3: 15.78, α 0.5: 19.80, α 0.7: 25.61. Choose α = 0.1; holdout MSE 17.22, a little worse than its build MSE, as honest tests usually are'},
    {"id": 'P3.4', "tier": 3, "group": 'Moving averages and exponential smoothing',
     "q": 'Smooth the heat-pump sales (23, 25, 27, 30, 32, 36, 36, 41) with α = 0.5, F2 = 23. Give the one-step errors for years 2–8 and their mean. What does the sign tell you, and what method fixes it?',
     "parts": ['error, year 2', 'error, year 3', 'error, year 4', 'error, year 5', 'error, year 6', 'error, year 7', 'error, year 8', 'mean error'],
     "answer": [2, 3.0, 4.5, 4.25, 6.125, 3.0625, 6.53125, 4.209821428571429], "tol": [0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005], "function": '—',
     "working": "Errors 2, 3, 4.5, 4.25, 6.125, 3.06, 6.53: all positive, mean ≈ 4.21. Smoothing always lags a trend; Holt's method (or a trend line) adds the slope back"},
    {"id": 'P3.5', "tier": 3, "group": 'Trend projection',
     "q": "FORECAST.LINEAR gives 58 for t = 10 and 61 for t = 12 on the same data. What are the trend line's slope and intercept, and its forecast for t = 15?",
     "parts": ['slope', 'intercept', 'forecast for t = 15'],
     "answer": [1.5, 43.0, 65.5], "tol": [0, 0, 0], "function": 'FORECAST.LINEAR',
     "working": 'slope = (61 − 58)/(12 − 10) = 1.5; intercept = 58 − 1.5 × 10 = 43; t = 15: 43 + 22.5 = 65.5'},
    {"id": 'P3.6', "tier": 3, "group": 'Seasonality and trend',
     "q": "In the retailer's data, take each year's fourth quarter minus its first quarter (years 1–4), and each year's fourth quarter divided by its first. Which is steadier, and what does that say about additive dummies versus a multiplicative model?",
     "parts": ['Q4 − Q1, year 1', 'Q4 − Q1, year 2', 'Q4 − Q1, year 3', 'Q4 − Q1, year 4', 'Q4 ÷ Q1, year 1', 'Q4 ÷ Q1, year 2', 'Q4 ÷ Q1, year 3', 'Q4 ÷ Q1, year 4'],
     "answer": [8.600000000000001, 8.100000000000001, 9.8, 9.5, 1.4725274725274726, 1.4111675126903553, 1.4827586206896552, 1.4203539823008848], "tol": [0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005], "function": '—',
     "working": 'Gaps 8.6, 8.1, 9.8, 9.5 ($ millions): they grow with the level. Ratios 1.47, 1.41, 1.48, 1.42: no drift. The seasonal effect is a percentage, not a fixed amount, so a multiplicative model suits it better than additive dummies'},
    {"id": 'P3.7', "tier": 3, "group": 'Time series decomposition',
     "q": "Use the retailer's decomposition (indexes 0.877, 0.984, 0.935, 1.205; trend on the deseasonalised series T = b0 + b1 t from all 17 quarters). Forecast the fourth quarter of year 5 (t = 20), and its growth on the same quarter a year earlier (32.1).",
     "parts": ['forecast, Y5 Q4 ($ millions)', 'growth on 32.1 (%, as a number)'],
     "answer": [33.94982637865446, 5.762698998923543], "tol": [0.005, 0.05], "function": 'TREND, AVERAGEIF',
     "working": 'T(20) = 20.514 + 0.3834 × 20 ≈ 28.18; × 1.205 ≈ 33.95; about 5.8 % above 32.1'},
    {"id": 'P3.8', "tier": 3, "group": 'Time series decomposition',
     "q": "A model's in-sample percentage errors have an RMS of 1.7 %; on a four-quarter holdout it is 5.3 %. Its forecast for next quarter is $26.6 million. Give an 80 % band using the holdout figure, and the (too narrow) band using the in-sample figure.",
     "parts": ['holdout band: lower ($ millions)', 'holdout band: upper', 'in-sample band: lower', 'in-sample band: upper'],
     "answer": [24.793268602895225, 28.406731397104778, 26.020482382060734, 27.179517617939272], "tol": [0.005, 0.005, 0.005, 0.005], "function": 'NORM.S.INV',
     "working": 'z for 80 % is 1.2816. Holdout: 26.6 × (1 ± 1.2816 × 0.053) ≈ 24.79 to 28.41. In-sample: ≈ 26.02 to 27.18, three times too confident: a model always fits the data it was built on better than the future'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": "Using years 1–3 only (months 1–36), compute the 12-month centred moving average and the monthly seasonal indexes (adjusted to average 1). Give January's and July's.",
     "parts": ["January's seasonal index", "July's seasonal index"],
     "answer": [1.218436321100648, 0.8641821716861559], "tol": [0.0005, 0.0005], "function": 'AVERAGE, AVERAGEIF',
     "working": 'January ≈ 1.218, July ≈ 0.864: January runs about 22 % above the trend, July about 14 % below'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": "Two tempting forecasts for year 4, made at the end of year 3: a straight trend line through months 1–36, and the same month last year. Give each one's MAPE on year 4.",
     "parts": ['trend line: MAPE on year 4 (%)', 'same month last year: MAPE on year 4 (%)'],
     "answer": [8.157449373322905, 9.140619429412972], "tol": [0.05, 0.05], "function": 'TREND, ABS, AVERAGE',
     "working": "Trend line ≈ 8.2 % (right on average, wrong every month); same month last year ≈ 9.1 % (right shape, misses a year's growth)"},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": 'Now the decomposition built on years 1–3: trend on the deseasonalised series times the index. Give its MAPE on year 4, and its MAPE on years 1–3 (the fit).',
     "parts": ['decomposition: MAPE on year 4 (%)', 'decomposition: MAPE on years 1–3 (%)'],
     "answer": [3.0889776013207264, 2.0246743398340965], "tol": [0.05, 0.05], "function": 'TREND, AVERAGEIF',
     "working": 'Holdout ≈ 3.1 %, fit ≈ 2.0 %: the best of the methods, and the holdout error, not the fit, is the honest one'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": 'A second question: fit trend + 11 month dummies (December the baseline) on years 1–3 with LINEST. Give its slope per month and its MAPE on year 4. Does it agree with the decomposition?',
     "parts": ['slope (tickets a month, per month)', 'MAPE on year 4 (%)'],
     "answer": [110.08333333333333, 3.107149669732844], "tol": [0.5, 0.05], "function": 'LINEST',
     "working": "Slope ≈ 110 tickets a month; holdout MAPE ≈ 3.1 %, close to the decomposition's 3.1 %: two honest methods agree"},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": "Refit the decomposition on all 48 months. Give the trend's slope (tickets a month per month) and the forecast for January and July of year 5 (months 49 and 55).",
     "parts": ['trend slope (tickets a month, per month)', 'forecast, January of year 5', 'forecast, July of year 5'],
     "answer": [125.68305141253974, 25768.527283178875, 18889.24893211927], "tol": [0.5, 5, 5], "function": 'TREND, AVERAGEIF',
     "working": 'Slope ≈ 126; January ≈ 25,769, July ≈ 18,889 tickets'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": "The holdout's percentage errors (year 4, decomposition) have an RMS of about 3.9 %. Give the 80 % band for January of year 5, and the agents needed to cover its upper edge at 800 tickets per agent.",
     "parts": ['80% band: lower (tickets)', '80% band: upper (tickets)', 'agents for January'],
     "answer": [24466.724364822425, 27070.33020153533, 34], "tol": [5, 5, 0], "function": 'NORM.S.INV, ROUNDUP',
     "working": '25,769 × (1 ± 1.2816 × 0.0394) ≈ 24,467 to 27,070; ROUNDUP(27,070/800) = 34 agents'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": 'Do the same for every month of year 5. Give the most and fewest agents needed, and the total agent-months, against staffing every month at the peak.',
     "parts": ['most agents in a month', 'fewest agents in a month', 'total agent-months', 'agent-months staffed at the peak all year'],
     "answer": [34, 25, 350, 408], "tol": [0, 0, 0, 0], "function": 'ROUNDUP, SUM, MAX',
     "working": 'Agents by month: 34, 31, 29, 27, 27, 25, 25, 27, 32, 32, 29, 32. Peak 34, low 25; 350 agent-months against 408 at a flat peak level'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Tallowfield Software',
     "q": 'Write the recommendation, anchored on numbers: a permanent team for the low months, and temporary agents for the peaks. How many permanent agents, and how many temporary agent-months?',
     "parts": ['permanent agents', 'temporary agent-months'],
     "answer": [25, 50], "tol": [0, 0], "function": 'SUM, MIN',
     "working": 'A permanent team of 25 covers the quietest month; temporary agents add 50 agent-months, most in January and from September to December. Limits: the forecast assumes next year looks like the last four (a product launch or a price change breaks it), and error grows with the horizon, so re-forecast every month and compare with actuals'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 35 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 9, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# ---- the course's methods, written out (the same as the practice set's) ----
def mape(actual, forecast):
    """Mean absolute percentage error, in percent."""
    a, f = np.asarray(actual, float), np.asarray(forecast, float)
    return 100 * np.mean(np.abs(a - f) / a)

def holt(y, alpha, beta):
    """Holt's method, started with L1 = Y1, b1 = Y2 − Y1. Returns the levels, the trends and the one-step forecasts
    (forecast[t] is made after period t + 1, for period t + 2)."""
    L, b = [float(y[0])], [float(y[1] - y[0])]
    for t in range(1, len(y)):
        f = L[-1] + b[-1]
        l = alpha * y[t] + (1 - alpha) * f
        b.append(beta * (l - L[-1]) + (1 - beta) * b[-1]); L.append(l)
    return np.array(L), np.array(b), np.array(L) + np.array(b)

def decompose(y, period):
    """The classical multiplicative decomposition: centred moving average, ratios Y/CMA, their average by season
    adjusted to average 1, the deseasonalised series and a straight trend through it. y: a pandas Series in time
    order. Returns (indexes, b0, b1, forecast function of t = 1, 2, …)."""
    y = pd.Series(np.asarray(y, float))
    ma = y.rolling(period).mean()                                      # ends at each row
    h = period // 2
    cma = (ma.shift(-(h - 1)) + ma.shift(-h)) / 2                      # centred on the row (two averages, averaged)
    season = np.arange(len(y)) % period
    raw = (y / cma).groupby(season).mean()
    idx = raw * period / raw.sum()
    t = np.arange(1, len(y) + 1)
    b1, b0 = np.polyfit(t, y / idx[season].values, 1)
    return idx.values, b0, b1, lambda tt: (b0 + b1 * np.asarray(tt)) * idx.values[(np.asarray(tt) - 1) % period]

## Tier 1 · Warm-up

One idea, one step, small numbers: naming a pattern, year-on-year changes, an error and its percentage, the mean
error and the MAE, the MSE and the RMSE, a moving average, one step of exponential smoothing, a trend line's forecast,
a forecast with quarter dummies, and a deseasonalised value.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Forecast errors are one `numpy` array: `e.mean()` is the mean error (the bias), `np.abs(e).mean()` the MAE,
`(e ** 2).mean()` the MSE and its square root the RMSE. `np.diff(y)` gives the changes from one period to the next.
Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
e = np.array([3, -5, 2, -4])                                      # P1.4 and P1.5
mean_error, mae_ = e.mean(), np.abs(e).mean()
mse_ = (e ** 2).mean()
print(f"mean error {mean_error}   MAE {mae_}   MSE {mse_}   RMSE {math.sqrt(mse_):.3f}")
assert agrees("P1.4", [mean_error, mae_]) and agrees("P1.5", [mse_, math.sqrt(mse_)])

orders = np.array([40, 44, 49, 52, 47, 45, 50, 56])               # P1.2: the changes
print("year-on-year changes:", np.diff(orders))
assert agrees("P1.2", list(np.diff(orders)) + ["D"])

assert agrees("P1.3", [120 - 112, 100 * abs(120 - 112) / 120])
assert agrees("P1.6", [np.mean([64, 70, 67])])                    # a three-week moving average
assert agrees("P1.7", [0.25 * 212 + 0.75 * 200])                  # one step of smoothing
assert agrees("P1.8", [35 + 1.8 * 11])
assert agrees("P1.9", [80 - 5, 80]) and agrees("P1.10", [46.2 / 1.10])

errors5 = np.array([4, -2, 3, -5, 1])                             # the course's warm-up: five errors
print("the warm-up's five errors: mean", errors5.mean(), " MAE", np.abs(errors5).mean(), " MSE", (errors5 ** 2).mean())
assert errors5.mean() == 0.2 and np.abs(errors5).mean() == 3 and (errors5 ** 2).mean() == 11

### Slide α: watch the forecast follow the demand

The course's twelve weeks of demand for a compressor part. Each week's forecast is α times last week's demand plus
1 − α times last week's forecast, starting from week 1's demand. With α = 0.2 the MSE over weeks 2–12 is 15.81; with
α = 0.5 it is 20.33: on a series that only wobbles about a level, a small α (heavy smoothing) wins. Slide α and
watch the gold line and the MSE.

In [ ]:
WEEKLY = [51, 46, 53, 48, 55, 47, 54, 49, 50, 45, 52, 56]

def smooth(y, alpha):
    """One-step forecasts F2 … F(n+1), with F2 = Y1."""
    f = [y[0]]
    for v in y[1:]: f.append(alpha * v + (1 - alpha) * f[-1])
    return np.array(f)

def smoothing_picture(alpha=0.2):
    f = smooth(WEEKLY, alpha)                                         # f[k] forecasts week k + 2
    err = np.array(WEEKLY[1:]) - f[:-1]
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.16)
    ax.plot(range(1, 13), WEEKLY, color=CHALK, marker="o", label="demand")
    ax.plot(range(2, 14), f, color=GOLD, marker="o", label="forecast")
    ax.set_xlim(0.5, 13.5); ax.set_ylim(40, 60); ax.set_xlabel("week"); ax.set_ylabel("units")
    ax.legend(loc="lower right", frameon=False)
    fig.suptitle(f"α = {alpha:.2f}: MSE over weeks 2–12 = {(err ** 2).mean():.2f}, week 13's forecast {f[-1]:.1f}",
                 x=0.07, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

_mse = lambda a: ((np.array(WEEKLY[1:]) - smooth(WEEKLY, a)[:-1]) ** 2).mean()
assert round(_mse(0.2), 2) == 15.81 and round(_mse(0.5), 2) == 20.33                    # the video's numbers

In [ ]:
interact(smoothing_picture, alpha=FloatSlider(value=0.2, min=0.05, max=0.95, step=0.05, description="α"));

## Tier 2 · Core

The module's standard methods, several steps each: the naive forecast's MAE, MSE and MAPE; moving averages against
the naive forecast; a weighted moving average; exponential smoothing with two values of α; a least-squares trend
line; one update of Holt's method; quarter dummies; centred moving averages and a seasonal-irregular ratio; and
seasonal indexes adjusted to average 1.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

### How you'd do this in Python

**Moving averages and smoothing.** With the data in a pandas `Series`, `demand.rolling(3).mean().shift(1)` is the
three-week moving-average forecast (the mean of the three weeks before), and `demand.shift(1)` the naive one.
`demand.ewm(alpha=0.2, adjust=False).mean()` is exponential smoothing: its value at week t is the forecast for week
t + 1, started from week 1's demand. Leave out `adjust=False` and pandas uses a different start.

**Trend and dummies.** `np.polyfit(t, y, 1)` fits a straight trend and returns the slope, then the intercept.
`np.linalg.lstsq(X, y)` does least squares with any columns: a column of 1s, the dummies and t; unlike LINEST, it
returns the coefficients in the order of the columns.

In [ ]:
demand = pd.Series(WEEKLY, index=range(1, 13), name="demand")
print("three-week moving-average forecasts:", demand.rolling(3).mean().shift(1).dropna().round(2).tolist()[:4], "…")
print("smoothed with α = 0.2:              ", demand.ewm(alpha=0.2, adjust=False).mean().round(2).tolist()[:4], "…")
assert demand.rolling(3).mean().shift(1).dropna().tolist()[:3] == [50, 49, 52]            # the video's numbers
assert [round(v, 6) for v in demand.ewm(alpha=0.2, adjust=False).mean().tolist()[:3]] == [51, 50, 50.6]

In [ ]:
pumps = [23, 25, 27, 30, 32, 36, 36, 41]                                                  # a heat-pump maker, years 1–8
t = np.arange(1, 9)
slope, intercept = np.polyfit(t, pumps, 1)
print(f"trend: {intercept:.2f} + {slope:.2f} t;  years 9 and 10: {intercept + 9 * slope:.2f}, {intercept + 10 * slope:.2f}")
assert round(slope, 9) == 2.5 and round(intercept, 9) == 20
L_, b_, F_ = holt(pumps, 0.5, 0.3)
print(f"Holt (α 0.5, β 0.3): level {L_[-1]:.2f}, trend {b_[-1]:.2f}; years 9 and 10: {F_[-1]:.2f}, {L_[-1] + 2 * b_[-1]:.2f}")
assert round(F_[-1], 2) == 42.58 and round(L_[-1] + 2 * b_[-1], 2) == 45.15

retail = [18.2, 21.9, 20.6, 26.8, 19.7, 22.2, 21.9, 27.8, 20.3, 22.9, 22.2, 30.1, 22.6, 26.4, 24.9, 32.1, 24.1]
q = np.arange(17) % 4                                                                     # 0 = Q1 … 3 = Q4
X = np.column_stack([np.ones(17), q == 0, q == 1, q == 2, np.arange(1, 18)]).astype(float)
coef = np.linalg.lstsq(X, np.array(retail), rcond=None)[0]
print("constant, Qtr1, Qtr2, Qtr3, t:", coef.round(4))
assert [round(v, 4) for v in coef] == [25.45, -7.845, -5.1, -6.425, 0.375]

In [ ]:
y = pd.Series([40, 44, 42, 45, 48])                                                       # P2.1: the naive forecast
err = (y - y.shift(1)).dropna()
assert agrees("P2.1", [err.abs().mean(), (err ** 2).mean(), 100 * (err.abs() / y[1:]).mean()])
y = pd.Series([36, 42, 39, 45, 42, 48])                                                   # P2.2: three-day MA
ma = y.rolling(3).mean()                                                                  # ma[k] forecasts day k + 2
mse_ma = ((y[3:6].values - ma[2:5].values) ** 2).mean()
mse_nv = ((y[3:6].values - y[2:5].values) ** 2).mean()
assert agrees("P2.2", list(ma[2:6]) + [mse_ma, mse_nv])
assert agrees("P2.3", [np.dot([0.5, 0.3, 0.2], [48, 42, 45])])                           # newest first
loads = pd.Series([120, 126, 117, 129])                                                   # P2.4
s3, s6 = loads.ewm(alpha=0.3, adjust=False).mean(), loads.ewm(alpha=0.6, adjust=False).mean()
mse3 = ((loads[1:].values - s3[:3].values) ** 2).mean()
mse6 = ((loads[1:].values - s6[:3].values) ** 2).mean()
assert agrees("P2.4", [s3[1], s3[2], s3[3], mse3, mse6])
b1_, b0_ = np.polyfit(range(1, 7), [12, 15, 17, 18, 23, 26], 1)                          # P2.5
assert agrees("P2.5", [b1_, b0_, b0_ + 7 * b1_])
L1 = 0.4 * 130 + 0.6 * (120 + 4); b1h = 0.2 * (L1 - 120) + 0.8 * 4                       # P2.6: one Holt update
assert agrees("P2.6", [L1, b1h, L1 + b1h, L1 + 3 * b1h])
ferry = pd.Series([30, 48, 75, 35, 34, 52, 79, 31])                                       # P2.7: no trend
qm = ferry.groupby(np.arange(8) % 4).mean()                                               # the quarter means
assert agrees("P2.7", [qm[3], qm[0] - qm[3], qm[1] - qm[3], qm[2] - qm[3], qm[2]])
r5 = pd.Series(retail[:5]); m4 = r5.rolling(4).mean().dropna().tolist()                  # P2.8
assert agrees("P2.8", [m4[0], m4[1], (m4[0] + m4[1]) / 2, retail[2] / ((m4[0] + m4[1]) / 2)])
raw = np.array([np.mean(v) for v in ([0.84, 0.86, 0.85], [1.04, 1.02, 1.03], [1.12, 1.10, 1.11], [1.06, 1.04, 1.05])])
ix = raw * 4 / raw.sum()                                                                  # P2.9
assert agrees("P2.9", list(ix) + [66.6 / ix[2]])
print("Every tier 2 answer, worked out with pandas and numpy, matches the practice set.")

### Holt's method: slide α and β

The heat-pump sales have a trend, so plain smoothing lags behind (every error positive, tier 3 shows it). Holt's
method keeps a level and a trend. With α = 0.5 and β = 0.3 it forecasts 42.58 for year 9 and 45.15 for year 10, close
to the trend line's 42.5 and 45.0. Slide α and β: a larger β lets the trend turn faster.

In [ ]:
def holt_picture(alpha=0.5, beta=0.3):
    L, b, F = holt(pumps, alpha, beta)
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.16)
    ax.plot(range(1, 9), pumps, color=CHALK, marker="o", label="sales")
    ax.plot(range(2, 10), F, color=SKY, marker="o", label="Holt's one-step forecasts")
    ax.plot([8, 9, 10], [L[-1], L[-1] + b[-1], L[-1] + 2 * b[-1]], color=SKY, ls="--")
    ax.plot(range(1, 11), 20 + 2.5 * np.arange(1, 11), color=VIOLET, lw=1.6, label="trend line 20 + 2.5t")
    ax.set_xlim(0.5, 10.5); ax.set_xlabel("year"); ax.set_ylabel("thousand units"); ax.legend(frameon=False)
    fig.suptitle(f"α = {alpha:.2f}, β = {beta:.2f}: year 9 → {L[-1] + b[-1]:.2f}, year 10 → {L[-1] + 2 * b[-1]:.2f}",
                 x=0.07, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

In [ ]:
interact(holt_picture, alpha=FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="α"),
         beta=FloatSlider(value=0.3, min=0.05, max=0.9, step=0.05, description="β"));

## Tier 3 · Stretch

Choose the method, compare on a holdout, spot the trap, work backwards: MAE against MSE when a big miss is costly, a
flexible curve that fits the past and fails the future, choosing α on the build weeks and judging it on the holdout,
smoothing that lags a trend, a trend line read back from two forecasts, additive against multiplicative seasons, a
forecast from a decomposition, and an honest range.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**A holdout.** Fit on the early periods, forecast the later ones, and score only those: the curve below fits weeks
1–8 better than their plain average, and misses weeks 9–12 by hundreds. `np.polyfit(t, y, 5)` is the six-coefficient
curve. **The decomposition** is `decompose` from the first cell, written out with pandas: `rolling(4).mean()` twice
shifted and averaged is the centred moving average, `groupby` averages the ratios by quarter. And `NormalDist().inv_cdf
(0.9)` from Python's `statistics` module is 1.2816, the z of an 80% band (a spreadsheet's NORM.S.INV(0.9)).

In [ ]:
build, hold = np.array(WEEKLY[:8]), np.array(WEEKLY[8:])
avg8 = build.mean()
curve = np.poly1d(np.polyfit(np.arange(1, 9), build, 5))
fit_ = [np.abs(build - avg8).mean(), np.abs(build - curve(np.arange(1, 9))).mean()]
out_ = [np.abs(hold - avg8).mean(), np.abs(hold - curve(np.arange(9, 13))).mean()]
print(f"MAE on weeks 1–8:  average {fit_[0]:.2f}, curve {fit_[1]:.2f}")
print(f"MAE on weeks 9–12: average {out_[0]:.2f}, curve {out_[1]:.1f}  (the curve says {curve(9):.0f} and {curve(10):.1f})")
assert agrees("P3.2", [fit_[0], fit_[1], out_[0], out_[1]])

fig, ax = plt.subplots(figsize=(11.5, 4.2))
fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.16)
tt = np.linspace(1, 10.2, 300)
ax.axvspan(8.5, 12.5, color=GHOST, alpha=0.6, lw=0); ax.text(10.5, 57.5, "holdout", color=DIM, ha="center")
ax.plot(tt, curve(tt), color=CORAL, label="six-coefficient curve")
ax.axhline(avg8, color=SKY, label="average of weeks 1–8")
ax.plot(range(1, 13), WEEKLY, color=CHALK, marker="o", ls="", label="demand")
ax.set_ylim(40, 60); ax.set_xlim(0.5, 12.5); ax.set_xlabel("week"); ax.set_ylabel("units"); ax.legend(frameon=False, loc="lower left")
fig.suptitle("The curve fits weeks 1–8 better, then dives off the chart in the holdout", x=0.07, ha="left",
             fontsize=15, color=GOLD)
footnote(fig); plt.show()

In [ ]:
act, A, B = np.array([52, 47, 55, 50]), np.array([52, 47, 54, 41]), np.array([49, 44, 52, 47])   # P3.1
assert agrees("P3.1", [np.abs(act - A).mean(), ((act - A) ** 2).mean(), np.abs(act - B).mean(), ((act - B) ** 2).mean()])
scores = {}
for a in (0.1, 0.3, 0.5, 0.7):                                                           # P3.3: choose α on weeks 2–8
    f = smooth(WEEKLY, a)[:-1]                                                            # forecasts of weeks 2–12
    e_ = np.array(WEEKLY[1:]) - f
    scores[a] = ((e_[:7] ** 2).mean(), (e_[7:] ** 2).mean())
best = min(scores, key=lambda a: scores[a][0])
print("α: (MSE weeks 2–8, MSE weeks 9–12):", {float(a): tuple(round(float(v), 2) for v in s) for a, s in scores.items()})
assert agrees("P3.3", [best, *scores[best]])
f = smooth(pumps, 0.5)[:-1]; e_ = np.array(pumps[1:]) - f                                 # P3.4: smoothing lags a trend
assert agrees("P3.4", list(e_) + [e_.mean()]) and (e_ > 0).all()
s_ = (61 - 58) / (12 - 10)                                                                # P3.5: back from two forecasts
assert agrees("P3.5", [s_, 58 - s_ * 10, 58 - s_ * 10 + s_ * 15])
r16 = np.array(retail[:16]).reshape(4, 4)                                                 # P3.6: rows are years
assert agrees("P3.6", list(r16[:, 3] - r16[:, 0]) + list(r16[:, 3] / r16[:, 0]))
idx_r, b0_r, b1_r, fc_r = decompose(retail, 4)                                            # P3.7
print("the retailer's indexes:", [round(float(v), 3) for v in idx_r], f" trend {b0_r:.2f} + {b1_r:.3f} t;  Y5 Q4 → {fc_r(20):.2f}")
assert [round(v, 3) for v in idx_r] == [0.877, 0.984, 0.935, 1.205]
assert agrees("P3.7", [fc_r(20), 100 * (fc_r(20) / 32.1 - 1)])
z80 = NormalDist().inv_cdf(0.9)                                                            # P3.8
assert agrees("P3.8", [26.6 * (1 - z80 * 0.053), 26.6 * (1 + z80 * 0.053), 26.6 * (1 - z80 * 0.017), 26.6 * (1 + z80 * 0.017)])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Tier 4 · The case: Tallowfield Software's support desk

Tallowfield Software makes business software (a fictional company; our own data). Its support desk answers the
tickets customers open. Here are four years of monthly tickets: a growing customer base, a January peak when licences
renew, a September peak after the holidays, and a quiet summer. The question: **how many support agents to roster in
each month of next year?** The plan, written before the analysis: build on years 1–3, hold out year 4 and choose the
method with the lowest MAPE on it, refit that method on all four years, and staff for the upper edge of an 80% band
(demand above it about one month in ten). One agent resolves 800 tickets a month.

| column | what it is |
|---|---|
| `month` | month number, 1–48 |
| `year` | year 1–4 |
| `month_name` | calendar month |
| `tickets` | support tickets opened that month |

The next cell builds the table from a seeded random generator, so everyone gets exactly the same 48 months (the
same as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
CASE_SEED, TICKETS_PER_AGENT = 3, 800
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def build_case(seed=CASE_SEED):
    """48 months of support tickets: a trend, a January and a September peak, and noise of about 3%."""
    r = random.Random(seed)
    shape = [1.20, 1.08, 1.01, 0.95, 0.92, 0.88, 0.86, 0.93, 1.10, 1.04, 0.97, 1.06]
    shape = [x / (sum(shape) / 12) for x in shape]
    tickets = [int(round((15000 + 130 * t) * shape[(t - 1) % 12] * (1 + r.gauss(0, 0.03)))) for t in range(1, 49)]
    return pd.DataFrame({"month": range(1, 49), "year": [1 + (t - 1) // 12 for t in range(1, 49)],
                         "month_name": [MONTHS[(t - 1) % 12] for t in range(1, 49)], "tickets": tickets})

df = build_case()

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n': 48,
 'total': 867265,
 'first3': [18208, 17099, 15110],
 'min': 13777,
 'max': 24006,
 'ses_mape': 8.9,
 'lin_mape': 8.2,
 'snaive_mape': 9.1,
 'dec_mape': 3.1,
 'reg_mape': 3.1,
 'agents': [34, 31, 29, 27, 27, 25, 25, 27, 32, 32, 29, 32]}
# @END STATS
S = CASE_STATS
assert len(df) == S["n"] and df["tickets"].sum() == S["total"] and df["tickets"].tolist()[:3] == S["first3"]
assert df["tickets"].min() == S["min"] and df["tickets"].max() == S["max"]
print(f"{len(df)} months, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the months in a DataFrame, each step of the case is a line or two. The cells below reproduce the video's case:
three tempting forecasts for year 4 (exponential smoothing, a straight trend line, the same month last year), then
the decomposition and the trend-plus-dummies regression, each judged on year 4 only.

In [ ]:
y = df["tickets"].astype(float)
train, test = y[:36].values, y[36:].values
# tempting 1: exponential smoothing, α chosen on years 1–3; its forecast for every month of year 4 is one flat level
best = None
for a in np.arange(1, 20) / 20:
    f = smooth(list(train), a)                                                            # forecasts of months 2–37
    m_ = ((train[1:] - f[:-1]) ** 2).mean()
    if best is None or m_ < best[0]: best = (m_, a, f[-1])
ses_f = np.full(12, best[2])
# tempting 2: a straight trend line through years 1–3; tempting 3: the same month last year
sl_, ic_ = np.polyfit(np.arange(1, 37), train, 1)
line_f = ic_ + sl_ * np.arange(37, 49)
snaive_f = train[24:36]
# the fix: the decomposition built on years 1–3
idx3, b0_3, b1_3, fc3 = decompose(train, 12)
dec_f = fc3(np.arange(37, 49))
# a second honest method: trend + 11 month dummies (December the baseline), least squares
mo = np.arange(36) % 12
X = np.column_stack([np.ones(36)] + [(mo == k).astype(float) for k in range(11)] + [np.arange(1, 37)])
bd = np.linalg.lstsq(X, train, rcond=None)[0]
reg_f = bd[0] + np.r_[bd[1:12], 0][np.arange(36, 48) % 12] + bd[-1] * np.arange(37, 49)
scores = {"smoothing (α = %.2f)" % best[1]: mape(test, ses_f), "trend line": mape(test, line_f),
          "same month last year": mape(test, snaive_f), "decomposition": mape(test, dec_f),
          "trend + month dummies": mape(test, reg_f)}
for k, v in scores.items(): print(f"MAPE on year 4, {k:22s}: {v:.1f}%")
assert [round(v, 1) for v in scores.values()] == [S["ses_mape"], S["lin_mape"], S["snaive_mape"], S["dec_mape"], S["reg_mape"]]
print("seasonal indexes (years 1–3):", dict(zip(MONTHS, [round(float(v), 3) for v in idx3])))

In [ ]:
# The optional cross-check: statsmodels' seasonal_decompose uses the same centred average and normalisation.
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    sd = seasonal_decompose(pd.Series(train), model="multiplicative", period=12)
    print("statsmodels' indexes:", sd.seasonal[:12].round(4).tolist())
    assert np.allclose(sd.seasonal[:12].values, idx3, atol=1e-9)
except ImportError:
    print("statsmodels isn't installed here: the hand-built indexes above are the ones the course uses.")

In [ ]:
# The decision: refit on all four years, forecast year 5, an 80% band from the holdout's % errors, and the agents
idx4, b0_4, b1_4, fc4 = decompose(y, 12)
f5 = fc4(np.arange(49, 61))
rms = math.sqrt(np.mean(((test - dec_f) / test) ** 2))                                   # the holdout's typical % error
z80 = NormalDist().inv_cdf(0.9)
lo5, up5 = f5 * (1 - z80 * rms), f5 * (1 + z80 * rms)
agents = [math.ceil(u / TICKETS_PER_AGENT) for u in up5]
print(f"holdout RMS % error {100 * rms:.1f}%, so the 80% band is ± {100 * z80 * rms:.1f}%")
print("agents by month:", dict(zip(MONTHS, agents)))
print(f"{sum(agents)} agent-months; staffing every month at the peak would take {12 * max(agents)}")
assert agents == S["agents"]

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.6, 1]})
fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.15, wspace=0.25)
ax.plot(range(1, 49), y, color=CHALK, lw=1.8, label="tickets")
ax.plot(range(49, 61), f5, color=SKY, label="year 5 forecast")
ax.fill_between(range(49, 61), lo5, up5, color=SKY, alpha=0.25, lw=0, label="80% band")
ax.set_ylim(0, 30000); ax.set_xlabel("month"); ax.set_ylabel("tickets"); ax.legend(frameon=False, loc="lower right")
low = min(agents)
ax2.bar(MONTHS, [low] * 12, color=CHALK, alpha=0.85)
ax2.bar(MONTHS, [a - low for a in agents], bottom=[low] * 12, color=SKY)
ax2.set_ylabel("agents"); ax2.tick_params(axis="x", labelsize=10)
fig.suptitle(f"A permanent team of {low}, plus {sum(agents) - 12 * low} temporary agent-months for the peaks",
             x=0.07, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: fit the regression with month dummies yourself, work out every
month's agents, and end with your recommendation and its numbers. Use the DataFrame `df` in a cell of your own (press
**+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas and numpy and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
mine = {
    "P4.1": [idx3[0], idx3[6]],
    "P4.2": [mape(test, line_f), mape(test, snaive_f)],
    "P4.3": [mape(test, dec_f), mape(train, fc3(np.arange(1, 37)))],
    "P4.4": [bd[-1], mape(test, reg_f)],
    "P4.5": [b1_4, f5[0], f5[6]],
    "P4.6": [lo5[0], up5[0], agents[0]],
    "P4.7": [max(agents), min(agents), sum(agents), 12 * max(agents)],
    "P4.8": [min(agents), sum(agents) - 12 * min(agents)],
}
for k, got in mine.items():
    assert agrees(k, [float(v) for v in got]), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(float(v), 4))}" for p, v in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")

## Sources
Every exercise, data set and company here is our own (Tallowfield Software is fictional). The methods are the
standard ones of a first course in business forecasting: forecast accuracy (MAE, MSE, MAPE), moving averages and
exponential smoothing, trend projection and Holt's linear trend method (C. C. Holt, 1957), seasonal dummy
variables, and the classical multiplicative decomposition; the definitions agree with Hyndman and Athanasopoulos,
Forecasting: Principles and Practice, 3rd edition, chapters 3, 5, 7 and 8 (otexts.com/fpp3, retrieved 2026-11-09).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs08_time_series/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs08_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert all((a == b) if isinstance(b, str) else abs(a - float(b)) < 1e-9
                   for a, b in zip(_q["answer"], _flat(_e["answer"]))) and len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    assert _P.CASE["tickets"] == df["tickets"].tolist() and _P.CASE["month_name"] == df["month_name"].tolist()
    print("matches the course's practice set")